In [2]:
!pip install -q rfdetr

In [2]:
pip install onnxruntime-gpu

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.8/280.8 MB 1.3 MB/s eta 0:00:0000:0100:07m
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import cv2
import pandas as pd
import numpy as np
from segment_anything import SamPredictor, sam_model_registry
import os 
from accuracy import compute_metrics
from generate_isolated_masks import generate_isolated_mask
from rfdetr import RFDETRBase
from PIL import Image

In [ ]:
# Load SAM model
sam = sam_model_registry["vit_b"]("/home/selc-a4-sr2/Solar_Rooftop_Detection/checkpoints/Final_Models/FineTune_model_3_epoch_20_03_03_2025.pth")
sam.to("cuda")
sam_predictor = SamPredictor(sam)

data_folder = "/home/selc-a4-sr2/Solar_Rooftop_Detection/Arial_validation_images/images"
mask_folder = "/home/selc-a4-sr2/Solar_Rooftop_Detection/Arial_validation_images/masks"
csv_path = "/home/selc-a4-sr2/Solar_Rooftop_Detection/Solar_Rooftop_Detection/Validation_results/validation_results_1_detr_SAM.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

results_data = []

checkpoint_path = "/home/selc-a4-sr2/rfdetrf.pth"  

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_wrapper = RFDETRBase()

underlying_model = model_wrapper.model
underlying_model.reinitialize_detection_head(num_classes=2)

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

pytorch_model = underlying_model.model
pytorch_model.load_state_dict(checkpoint['model'])

pytorch_model.to(device)
pytorch_model.eval()

for img_name in os.listdir(data_folder):
    
    img_path = os.path.join(data_folder, img_name)
    image = cv2.imread(img_path)
    
    detections = model_wrapper.predict(image, threshold=0.5)
    boxes = detections.xyxy
    
    mask_path = os.path.join(mask_folder, img_name)
    gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    gt_mask = torch.from_numpy(gt_mask).to("cuda")  # Convert to torch tensor on GPU
    
    sam_predictor.set_image(image)
    
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        
        isolated_mask = generate_isolated_mask(gt_mask, [x1, y1, x2, y2])
        
        masks, _, _ = sam_predictor.predict(box=np.array([x1, y1, x2, y2]))
            
        pred_mask = masks[0]
        pred_mask = torch.from_numpy(masks[0]).to("cuda") 
        metrics = compute_metrics(pred_mask, isolated_mask)

        results_data.append(metrics)
        
metrics_df = pd.DataFrame(results_data, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall", "region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
metrics_df.to_csv(csv_path, index=False)

Loading pretrain weights


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

metrics = pd.read_csv("/home/selc-a4-sr2/Solar_Rooftop_Detection/Solar_Rooftop_Detection/Validation_results/validation_results_1_detr_SAM.csv")


In [2]:
metrics

,pixel_iou,pixel_dice,pixel_accuracy,pixel_precision,pixel_recall,region_iou,region_dice,region_precision,region_recall,region_success_accuracy
0,0.908247,0.951918,0.998803,0.922820,0.982910,0.908247,0.951918,0.922820,0.982910,1.0
1,0.837329,0.911463,0.998880,0.847783,0.985486,0.837329,0.911463,0.847783,0.985486,1.0
2,0.926753,0.961984,0.999333,0.936269,0.989151,0.926753,0.961984,0.936269,0.989151,1.0
3,0.944333,0.971369,0.999842,0.998936,0.945284,0.944333,0.971369,0.998936,0.945284,1.0
4,0.786678,0.880604,0.999517,0.794381,0.987824,0.786678,0.880604,0.794381,0.987824,1.0
...,...,...,...,...,...,...,...,...,...,...
13643,0.685907,0.813695,0.999600,1.000000,0.685907,0.685907,0.813695,1.000000,0.685907,1.0
13644,0.723887,0.839831,0.999710,0.731864,0.985167,0.723887,0.839831,0.731864,0.985167,1.0
13645,0.769577,0.869786,0.999169,0.875414,0.864231,0.769577,0.869786,0.875414,0.864231,1.0
13646,0.747298,0.855376,0.999710,0.824015,0.889219,0.747298,0.855376,0.824015,0.889219,1.0


In [3]:
output = metrics.mean(axis=0)
output

pixel_iou                  0.744573
pixel_dice                 0.840372
pixel_accuracy             0.998134
pixel_precision            0.819180
pixel_recall               0.901531
region_iou                 0.744573
region_dice                0.840372
region_precision           0.819180
region_recall              0.901531
region_success_accuracy    0.953326
dtype: float64